# 18b — score ONE epoch of rung 18Run once per epoch (`-p EPOCH 1|2|3`). **Every epoch is evaluated** (RULES §6b): rung 06never benchmarked its own ep3 while `eval_loss` was still falling, and rungs 14 and 15 bothended up comparing their ep3 against rung 06's **ep2** — an unevaluated epoch is a missingcontrol, not a discarded one.## What is read, in order1. 🆕 **Spearman r** of the predicted count vs gold on the `Clips` template — the run's   pre-registered win metric. Exact-match accuracy cannot separate a model that learned the   gold's marginal distribution from one that reads the frame; a rank metric can.2. **`margin_OOD`** — pre-registered as a NO-FALL condition. A rise in r bought by giving up   OOD margin is not a win.3. `bucket_mean` against rung 06 ep3's **0.5724**.The rung-06 control is not hard-coded: it is recomputed here from rung 06 ep3's own stored`predictions.json` through the SAME `frame.metrics` code path, so the comparison cannot driftagainst a number copied out of a note.⚠️ Those stored answers were produced on a different card, and ~0.5% of archived answerschange on a GPU swap. That is far below the deltas this rung is looking for, but it is whythe control is labelled *archived* wherever it is printed.

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, and it must be THIS interpreter's bin.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models", EXP / "_tools",
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME must be set BEFORE the offline flags mean anything. The SDK's judge
# (`Qwen/Qwen3-4B`, ~7.6 GB) is cached at /workspace/hf_cache, NOT at the default
# ~/.cache/huggingface — which is empty on this pod. With OFFLINE set and HF_HOME unset,
# transformers looks in the empty default cache and raises LocalEntryNotFoundError from
# inside `run_baseline`, i.e. AFTER the 17 GB merge and the whole inference pass. Rung 06's
# 06c never hit this because it did not set the offline flags at all and simply downloaded.
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
from count_aug_train import CountAugConfig, list_checkpoints, merge_checkpoint
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "| repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
SMOKE = True          # True -> 40 questions, wiring only. Full: -p SMOKE False
EPOCH = 1             # 1 | 2 | 3 — resolved against list_checkpoints, not a step number
RUN   = "18_count_aug_v1"

KEEP_MERGED = False   # a merged checkpoint is ~17 GB; keep only the one we ship
DATA_ROOT   = "/workspace/orena-data"
RUNG06_EP3  = "/workspace/repo/experiments/06-vit-lora/runs/06_vit_lora_v1/ep3_full"
REF_BUCKET_MEAN_06EP3 = 0.5724


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
RUN_DIR = EXP / "runs" / RUN
RUN_TAG = f"ep{EPOCH}_smoke" if SMOKE else f"ep{EPOCH}_full"

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

cfg = CountAugConfig(exp_dir=EXP, run_name=RUN, data_root=DATA_ROOT)
CKPTS = list_checkpoints(cfg)
assert 1 <= EPOCH <= len(CKPTS), f"EPOCH {EPOCH} but only {len(CKPTS)} checkpoints: {[c.name for c in CKPTS]}"
CKPT = CKPTS[EPOCH - 1]

print(f"SMOKE   {SMOKE}")
print(f"epoch   {EPOCH}/{len(CKPTS)} -> {CKPT.name}")
print(f"run_dir {RUN_DIR}")
print(f"tag     {RUN_TAG}")


In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ---
# `run_baseline` loads the judge only AFTER the 17 GB merge and the full inference pass, so
# a missing judge cache fails ~45 minutes in with everything already paid for. This is that
# failure, hoisted to the front and made cheap: the tokenizer alone proves the cache resolves
# under the offline flags. RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. The judge is cached at /workspace/hf_cache "
        "on this pod, not at the default ~/.cache/huggingface. Fix the env — do NOT disable "
        "the offline flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")


In [ ]:
# --- merge the adapter (per-epoch, namespaced so merges never overwrite) ---------
merged = cfg.merged_dir / CKPT.name
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
    MERGED_HERE = False
else:
    t0 = time.perf_counter()
    merged = merge_checkpoint(cfg, CKPT)
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter() - t0:.0f}s -> {merged}")
merged


In [ ]:
# --- eval -----------------------------------------------------------------------
t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT, model_path=merged, out_dir=RUN_DIR, run_name=RUN_TAG,
        max_pixels=1280 * 720, seed=42, n_eval=40 if SMOKE else None,
    )
    # rung 15's post-processor is a SECOND variable. Rung 18's levers are DATA; the
    # inference path must stay rung 06's exactly or the delta is not attributable.
    assert cfg_eval.answer_postprocess is None, "answer_postprocess must stay None"
    assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter() - t0) / 60:.1f} min")
finally:
    if MERGED_HERE and not KEEP_MERGED and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")


In [ ]:
# --- canonical scoring + gates (all RAISE) --------------------------------------
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")

missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; every margin would be inflated"

metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# The mode gate, on the ARTIFACT rather than the variable: `-p SMOKE False` failing to take
# effect is otherwise indistinguishable from a successful full run.
assert (len(res) < 1000) == SMOKE, (
    f"MODE GATE FAILED: SMOKE={SMOKE} but the eval scored {len(res)} rows")
if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"

print("gates OK | bucket_mean:", round(strat["bucket_mean"], 4),
      "| margin_OOD:", round(strat["margin_OOD"], 4))


In [ ]:
# --- 🆕 THE HEADLINE: count discrimination on the `Clips` template ---------------
# Pre-registered: a win needs r to RISE and margin_OOD not to fall. The rung-06 control is
# recomputed from its own archived predictions through this same function, never copied.
preds = metrics.predictions_frame(RUN_DIR / RUN_TAG)
rank = metrics.count_rank_report(preds, gold)

ref_preds = metrics.predictions_frame(Path(RUNG06_EP3))
ref_rank = metrics.count_rank_report(ref_preds, gold)

rows = []
for cell in ("pooled", "ID", "OOD"):
    a, b = ref_rank[cell], rank[cell]
    rows.append({
        "cell": cell, "n": b["n"],
        "r_06ep3_archived": round(a["r"], 4), "r_18": round(b["r"], 4),
        "delta_r": round(b["r"] - a["r"], 4),
        "ci_18": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]",
        "bias_06": round(a["bias"], 3), "bias_18": round(b["bias"], 3),
        "exact_06": round(a["exact"], 4), "exact_18": round(b["exact"], 4),
        "unreadable_18": b["n_unreadable"],
    })
rank_df = pd.DataFrame(rows)
print(rank_df.to_string(index=False))
print()
print("⚠️  The 0.43 quoted in gold-is-signal-model-underuses-it is NOT this quantity — it was")
print("    measured on 40 frames restricted to gold 3–6, and range restriction attenuates a")
print("    rank correlation toward zero by construction. The control above is the SAME")
print("    template, the SAME range and the SAME code path as the arm it is compared with.")


In [ ]:
# --- the second cell: margin, buckets, and the run's row ------------------------
bf = pd.DataFrame(strat["by_format"])
num = bf[bf.answer_format == "number"].set_index("distribution")
row = {
    "run": RUN, "epoch": EPOCH, "checkpoint": CKPT.name,
    "bucket_mean": strat["bucket_mean"],
    "acc_ID": strat["acc_ID"], "acc_OOD": strat["acc_OOD"],
    "margin_ID": strat["margin_ID"], "margin_OOD": strat["margin_OOD"],
    "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
    "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
    "clips_r_pooled": rank["pooled"]["r"], "clips_r_ID": rank["ID"]["r"],
    "clips_r_OOD": rank["OOD"]["r"], "clips_bias": rank["pooled"]["bias"],
    "d_bucket_mean_vs_06ep3": strat["bucket_mean"] - REF_BUCKET_MEAN_06EP3,
    "d_clips_r_vs_06ep3": rank["pooled"]["r"] - ref_rank["pooled"]["r"],
}
print(pd.DataFrame([row]).T.to_string(header=False))

# 🔴 The pre-registered verdict, printed as a verdict and not left to the reader.
_r_up = row["d_clips_r_vs_06ep3"] > 0
_ood_held = row["margin_OOD"] >= 0.0
print(f"\nPRE-REGISTERED READ  r rises: {_r_up}  |  margin_OOD holds: {_ood_held}  "
      f"-> {'WIN' if (_r_up and _ood_held) else 'NOT A WIN'}")
print("A faithful negative is a real result and is recorded as one.")


In [ ]:
# --- persist: RESULTS.csv + the ledger (full runs only) -------------------------
if not SMOKE:
    out = EXP / "RESULTS.csv"
    df = pd.concat([pd.read_csv(out), pd.DataFrame([row])], ignore_index=True) if out.exists() \
        else pd.DataFrame([row])
    df.to_csv(out, index=False)
    rank_df.to_csv(EXP / f"RESULTS_rank_ep{EPOCH}.csv", index=False)
    ledger.register_run(RUN_DIR / RUN_TAG, strat, experiment="18-count-aug",
                        run=f"{RUN}__{RUN_TAG}",
                        model=f"18 count-aug epoch {EPOCH} ({CKPT.name})", date="2026-07-28")
    print("wrote", out)
else:
    print("SMOKE — nothing persisted to RESULTS.csv or the ledger")


In [ ]:
# --- eyeball: what did it actually SAY on the counting questions? ---------------
# (user rule: a results summary AND error examples on every run)
g = gold.copy()
g["template"] = g["question"].map(metrics.template_of)
clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
look = clips.merge(preds, on="qID").head(400)
look["gold_n"] = look["answer"].map(metrics.read_count)
look["pred_n"] = look["prediction"].map(metrics.read_count)
look["err"] = look["pred_n"] - look["gold_n"]

print("--- predicted-vs-gold crosstab (Clips) ---")
print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())
print("\n--- the worst misses ---")
print(look.reindex(look["err"].abs().sort_values(ascending=False).index)
          .head(12)[["qID", "gold_n", "pred_n", "prediction"]].to_string(index=False))


## After all three epochs1. Select by **acc_OOD**, per-epoch — never last-epoch-by-default, and discard any checkpoint   that wins ID while dropping OOD (RULES §6).2. Read L1 with probe **16a** (`16a_zero_probe.ipynb`) pointed at the selected checkpoint:   zero-emission on the held-out slice, ABSENT **and** PRESENT arms — never ABSENT alone.3. Read L2 with probe **16d** (`16d_format_audit.ipynb`): illegal rate on `number`, with   `binary` / `fo_class` as the within-run control this rung never touched.